# Table of Contents

- [Case Study](#Case-Study)
- [Overview](#Overview)

## 1. Data Loading
- [Initial Data Review](#Initial-Data-Review)
- [Summary](#Summary)

## 2. Data Review and Storage
- [Data Review](#Data-Review)
  - [Missing Value Assessment](#Missing-Value-Assessment)
  - [Duplicate Keys Assessment](#Duplicate-Keys-Assessment)
  - [Review of Duplicate Product Records - Findings and Decision](#Review-of-Duplicate-Product-Records---Findings-and-Decision)
  - [Format Validation](#Format-Validation)
  - [Decimal Format Validation](#Decimal-Format-Validation)
  - [Trailing/Leading Whitespace Assessment](#Trailingleading-whitespace-assessment)
  - [Data Review Summary](#Data-Review-Summary)
- [Data Standardisation](#Data-Standardisation)
  - [Resolve Duplicate Product Records](#Resolve-Duplicate-Product-Records)
  - [Handle Invalid Date Values](#Handle-Invalid-Date-Values)
- [Assign Data Types](#Assign-Data-Types)
- [Data Quality Validation](#Data-Quality-Validation)
  - [Primary Key Validation](#Primary-Key-Validation)
  - [Referential Integrity](#Referential-Integrity)
  - [Primary and Foreign Keys Identification](#Primary-and-Foreign-Keys-Identification)
  - [Date Validation](#Date-Validation)
  - [Numeric Range Validation](#Numeric-Range-Validation)
  - [Negative Order Quantity: Findings](#Negative-Order-Quantity-Findings)
  - [Record Count Reconciliation](#Record-Count-Reconciliation)
  - [Summary](#Summary)

## 3. Product Master Transformations
- [Transformation Validation](#Transformation-Validation)

## 4. Sales Order Transformations
- [Join the Tables](#Join-the-tables)
- [Calculate LeadTimeInBusinessDays](#Calculate-LeadTimeInBusinessDays)
- [Calculate TotalLineExtendedPrice](#Calculate-TotalLineExtendedPrice)
- [Rename Freight](#Rename-Freight)
- [Create Publish Orders Dataset](#Create-Publish-Orders-Dataset)
- [Validation](#Validating-the-output)

## 5. Analysis Questions
- [Analysis 1](#Analysis-1)
- [Analysis 2](#Analysis-2)

# Case Study

**Name:** Bukola Akinsola  
**Date:** 6 July 2026

## Overview

This notebook documents the end-to-end solution for the interview case study. It covers data loading, initial data review, data cleaning and standardisation, data quality validation, the required business transformations, and the final analysis.

The notebook is organised to reflect a typical data engineering workflow, with each stage documenting the approach taken, the decisions made, and the validation performed before moving to the next step.

# 1. Data Loading

The first step is to load the source datasets into Spark without applying any transformations or schema inference.

All columns are initially loaded as strings (`inferSchema=False`) to preserve the raw data exactly as received. This allows the data to be reviewed and validated before assigning data types.

In [1]:
#import libraries
# Creates the Spark application and provides the entry point for working with DataFrames
from pyspark.sql import SparkSession

# Contains built-in Spark SQL functions used for transformations, aggregations and calculations
from pyspark.sql import functions as F

# Provides window functions for operations such as ranking, running totals and row numbering
from pyspark.sql.window import Window

# Used to repeatedly apply an operation across multiple DataFrames (e.g., unioning multiple DataFrames)
from functools import reduce

# Spark data types used when explicitly assigning column data types
from pyspark.sql.types import IntegerType, DoubleType, BooleanType


In [ ]:
# Create a Spark session
spark = (
    SparkSession.builder
    .appName("TakeHomeAssessment")
    .getOrCreate()
)

# Folder containing the input CSV files
DATA_DIR = "./data"

# Helper function to read a CSV file with consistent options
def read_csv(file_name):
    return (
        spark.read
        .format("csv")
        .option("header", True)         # First row contains column names
        .option("quote", '"')           # Handle quoted values
        .option("escape", '"')          # Handle escaped quotes within values
        .option("inferSchema", False)   # Read all columns as strings; assign data types later
        .csv(f"{DATA_DIR}/{file_name}")
    )

In [5]:
#call the function to read the source files into DataFrames with the raw_ prefix

raw_product = read_csv("products.csv")
raw_sales_order_header = read_csv("sales_order_header.csv")
raw_sales_order_detail = read_csv("sales_order_detail.csv")

## Initial Data Review

**Question being answered:** *What issues exist in the raw data?*

Before performing any transformations, the datasets are reviewed to gain an understanding of their structure and contents.

This review helps identify the expected data types, understand the available columns, and detect any obvious data quality issues.

In [ ]:
#check first five rows of each df

raw_product.show(5, truncate=False)

raw_sales_order_header.show(5, truncate=False)

raw_sales_order_detail.show(5, truncate=False)

In [7]:
#print schema to see the current schema of the raw df
raw_product.printSchema()

raw_sales_order_header.printSchema()

raw_sales_order_detail.printSchema()

root
 |-- ProductID: string (nullable = true)
 |-- ProductDesc: string (nullable = true)
 |-- ProductNumber: string (nullable = true)
 |-- MakeFlag: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- SafetyStockLevel: string (nullable = true)
 |-- ReorderPoint: string (nullable = true)
 |-- StandardCost: string (nullable = true)
 |-- ListPrice: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- SizeUnitMeasureCode: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- WeightUnitMeasureCode: string (nullable = true)
 |-- ProductCategoryName: string (nullable = true)
 |-- ProductSubCategoryName: string (nullable = true)

root
 |-- SalesOrderID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- ShipDate: string (nullable = true)
 |-- OnlineOrderFlag: string (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- SalesPersonID: string (nullable = true)
 |-- Freight: strin

In [9]:
#check the number or rows of each df

print(f"raw_products has {raw_product.count()} rows")
print(f"raw_sales_order_header has {raw_sales_order_header.count()} rows")
print(f"raw_sales_order_detail has {raw_sales_order_detail.count()} rows")

raw_products has 303 rows
raw_sales_order_header has 31465 rows
raw_sales_order_detail has 121317 rows


#### Summary

The source data has been successfully loaded into the following raw tables:

- `raw_product`
- `raw_sales_order_header`
- `raw_sales_order_detail`

Initial data assessment has also been completed, including checks for missing values, duplicate records, and date and numeric formatting issues. These findings informed the data cleaning and transformation steps in the next stage.

# 2. Data Review and Storage

**Question being answered:** *How do I prepare the data for downstream use*

## Data Review

The datasets are assessed before assigning data types.

At this stage, the objective is to identify issues that could affect data type conversion or indicate inconsistencies in the raw source data.

The following checks are performed:


- Missing values Assessment
- Duplicate keys Assessment
- Date format validation
- Decimal Format Validation
- Trailing/leading whitespace assessment

Any issues identified during this stage will be considered during data standardisation.

### Missing Value Assessment

Identify columns containing NULL values in the raw datasets.

This assessment helps determine whether missing values require handling during transformation or should be retained based on business requirements.

In [13]:
# Helper function to check nulls in all DataFrames

def check_nulls(df):
    return df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

In [ ]:
check_nulls(raw_product).show()

check_nulls(raw_sales_order_header).show()

check_nulls(raw_sales_order_detail).show()

In [17]:
# Inspect rows with missing values to understand the pattern before applying any transformations

product_null_review = reduce(
    lambda x, y: x | y,
    [F.col(c).isNull() for c in raw_product.columns]
)

In [ ]:
raw_product.filter(product_null_review).show(20, truncate=False)

### Duplicate Keys Assessment

Identify duplicate records in the raw datasets, with particular attention to primary key columns.

This assessment determines whether duplicate records represent data quality issues or valid business scenarios, and informs the appropriate handling strategy before applying transformations.

In [23]:
def check_duplicate_ids(df, id_columns):
    return (
        df.groupBy(id_columns)
        .count()
        .filter(F.col("count") > 1)
    )

In [25]:
check_duplicate_ids(raw_product, ["ProductID"]).show()

check_duplicate_ids(raw_sales_order_header, ["SalesOrderID"]).show()

check_duplicate_ids(raw_sales_order_detail,["SalesOrderDetailID"]).show()

+---------+-----+
|ProductID|count|
+---------+-----+
|      714|    2|
|      716|    2|
|      715|    2|
|      713|    2|
|      882|    2|
|      883|    2|
|      884|    2|
|      881|    2|
+---------+-----+

+------------+-----+
|SalesOrderID|count|
+------------+-----+
+------------+-----+

+------------------+-----+
|SalesOrderDetailID|count|
+------------------+-----+
+------------------+-----+



In [27]:
duplicate_product_ids = check_duplicate_ids(raw_product, ["ProductID"])

In [ ]:
#review duplicate rows to investigate further

raw_product.join(
    duplicate_product_ids.select("ProductID"),
    "ProductID",
    "inner"
).show(truncate=False)

#### Review of Duplicate Product Records - Findings and Decision

The duplicate `ProductID` records were examined to determine whether they represented duplicate data or distinct products.

The review showed that all duplicate records contained identical values across all attributes except `ProductCategoryName` and `ProductSubCategoryName`. In each duplicate pair, one record contained populated category information while the other contained missing category information. Additionally, some duplicate pairs had differing subcategory labels (for example, *Shirt* and *Jersey*).

As `ProductID` is expected to uniquely identify a single product, these differences were treated as data quality inconsistencies rather than separate products. There was insufficient information to determine which subcategory label was more accurate; therefore, the decision was based on data completeness rather than business interpretation.

**Decision**

For the standardisation step, the record containing populated `ProductCategoryName` information will be retained, as it provides a more complete classification for downstream transformations and analysis.

### Format Validation

Before changing data types, the date and numeric columns are reviewed for any formatting issues. This helps catch invalid values before they are converted.

In [33]:
# Helper function to check data format

def check_format(df, column, pattern):
    return (
        df.filter(
            F.col(column).isNotNull() &
            ~F.col(column).rlike(pattern)
        )
        .select(column)
        .distinct()
    )

In [35]:
date_pattern = r"^\d{4}-\d{2}-\d{2}$"


#check for the columns with date values in the raw_sales_order_header:OrderDate and ShipDate

check_format(
    raw_sales_order_header,
    "OrderDate",
    date_pattern
).show(truncate=False)


check_format(
    raw_sales_order_header,
    "ShipDate",
    date_pattern
).show(truncate=False)

+---------+
|OrderDate|
+---------+
|2021-06  |
+---------+

+--------+
|ShipDate|
+--------+
+--------+



**Findings**

A small number of records contain `OrderDate` values in the format `yyyy-MM` (for example, `2021-06`) instead of the expected `yyyy-MM-dd` format. As only two components are present, it is not possible to determine whether the value represents a year and month with the day missing, or whether it is an incorrectly formatted date. The intended date cannot be determined from the source data alone.

**Decision**

The records were retained, but the invalid `OrderDate` values were converted to `NULL` during data type assignment.

**Justification**

The missing day cannot be inferred from the source data without making an assumption. Converting these values to `NULL` preserves the original data, makes the data quality issue explicit, and avoids introducing incorrect dates that could affect downstream calculations such as lead time and yearly analysis.

### Decimal Format Validation

Columns expected to contain decimal values are checked to confirm that they follow a valid numeric format before data types are assigned.

This helps identify formatting issues early and ensures the values can be converted to numeric data types without introducing errors during casting.

In [37]:
# Match valid decimal values (e.g. 10, 10.50, .25)
decimal_pattern = r"^(\d+(\.\d+)?|\.\d+)$"

# Check that the specified columns contain values in a valid decimal format
def validate_decimal_columns(df, table_name, columns):
    print(f"\nValidating decimal columns in {table_name}")
    
    for column in columns:
        print(f"\nChecking {column}...")
        check_format(df, column, decimal_pattern).show(truncate=False)

In [39]:
validate_decimal_columns(
    raw_product,
    "Product",
    ["StandardCost", "ListPrice", "Weight"]
)

validate_decimal_columns(
    raw_sales_order_header,
    "SalesOrderHeader",
    ["Freight"]
)

validate_decimal_columns(
    raw_sales_order_detail,
    "SalesOrderDetail",
    ["UnitPrice", "UnitPriceDiscount"]
)


Validating decimal columns in Product

Checking StandardCost...
+------------+
|StandardCost|
+------------+
+------------+


Checking ListPrice...
+---------+
|ListPrice|
+---------+
+---------+


Checking Weight...
+------+
|Weight|
+------+
+------+


Validating decimal columns in SalesOrderHeader

Checking Freight...
+-------+
|Freight|
+-------+
+-------+


Validating decimal columns in SalesOrderDetail

Checking UnitPrice...
+---------+
|UnitPrice|
+---------+
+---------+


Checking UnitPriceDiscount...
+-----------------+
|UnitPriceDiscount|
+-----------------+
+-----------------+




### Trailing/leading whitespace assessment

Check text columns for leading and trailing whitespace that could affect comparisons or grouping.
The assessment identified whitespace in the unit measure code columns, which was removed during the cleaning stage.

In [ ]:
# Check for leading or trailing whitespace in unit measure codes
raw_product.filter(
    (F.col("SizeUnitMeasureCode") != F.trim(F.col("SizeUnitMeasureCode"))) |
    (F.col("WeightUnitMeasureCode") != F.trim(F.col("WeightUnitMeasureCode")))
).show(truncate=False)

In [49]:
# Remove leading and trailing whitespace from the unit measure code columns
raw_product = (
    raw_product
    .withColumn(
        "SizeUnitMeasureCode",
        F.trim("SizeUnitMeasureCode")
    )
    .withColumn(
        "WeightUnitMeasureCode",
        F.trim("WeightUnitMeasureCode")
    )
)

In [51]:
# Confirm that no leading or trailing whitespace remains
raw_product.filter(
    (F.col("SizeUnitMeasureCode") != F.trim(F.col("SizeUnitMeasureCode"))) |
    (F.col("WeightUnitMeasureCode") != F.trim(F.col("WeightUnitMeasureCode")))
).count()

0

### Data Review Summary

The raw data has now been assessed for basic data quality issues that may affect downstream processing.

The assessment included:


- Missing value assessment
- Duplicate business key assessment
- Date format validation
- Numeric format validation
- Trailing/leading whitespace assessment

The findings from this assessment will be used to guide the data standardisation process, where appropriate data types will be assigned and the cleaned datasets prepared for further validation and transformation.

## Data Standardisation

The data quality assessment identified a small number of issues in the raw source data that should be addressed before the data is stored for downstream processing.

The standardisation process includes:

- Resolving duplicate Product records.
- Handling malformed date values e.g: ```2021-06```
- Assigning appropriate data types.

The resulting datasets will be stored as the `store_` layer and used for subsequent data quality validation and business transformations.

### Resolve Duplicate Product Records

The duplicate assessment identified multiple records sharing the same `ProductID`.

A review of these records showed that one record consistently contained populated product category information while the other contained missing category values. As `ProductID` uniquely identifies a product, these records were treated as duplicate representations of the same product.

To maximise data completeness, the record containing `ProductCategoryName` information is retained.

In [59]:
# Rank duplicate ProductID records so the most complete record is kept
window = (
    Window
    .partitionBy("ProductID")
    .orderBy(
        F.col("ProductCategoryName").isNull(),      # Prefer rows with a category
        F.col("ProductSubCategoryName").isNull()    # Then prefer rows with a subcategory
    )
)

# Keep the highest-ranked record for each ProductID
store_product = (
    raw_product
    .withColumn("row_num", F.row_number().over(window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

### Handle Invalid Date Values

During the data review, a small number of `OrderDate` values were found to be stored in an incomplete format (`yyyy-MM`) rather than the expected `yyyy-MM-dd`.

As the missing day component cannot be inferred with confidence, these values are treated as invalid and replaced with `NULL` prior to data type conversion.

In [64]:
# Match dates in the expected yyyy-MM-dd format
date_pattern = r"^\d{4}-\d{2}-\d{2}$"

# Retain only valid date values; invalid formats are set to NULL
store_sales_order_header = (
    raw_sales_order_header
    .withColumn(
        "OrderDate",
        F.when(
            F.col("OrderDate").rlike(date_pattern),
            F.col("OrderDate")
        )
    )
    .withColumn(
        "ShipDate",
        F.when(
            F.col("ShipDate").rlike(date_pattern),
            F.col("ShipDate")
        )
    )
)

## Assign Data Types

The cleaned datasets have now been standardised by resolving duplicate records, handling malformed dates, trimming whitespace, and assigning appropriate data types. These datasets are stored as:

- `store_product`
- `store_sales_order_header`
- `store_sales_order_detail`

These tables are used as the input for the data quality validation stage.

In [68]:
#convert non string datatypes to the correct data types. string columns are not highlighted here

store_product = (
    store_product
    .fillna({"Color": "N/A"})
    .withColumn("ProductID", F.col("ProductID").cast(IntegerType()))
    .withColumn("MakeFlag", F.col("MakeFlag").cast(BooleanType()))
    .withColumn("SafetyStockLevel", F.col("SafetyStockLevel").cast(IntegerType()))
    .withColumn("ReorderPoint", F.col("ReorderPoint").cast(IntegerType()))
    .withColumn("StandardCost", F.col("StandardCost").cast(DoubleType()))
    .withColumn("ListPrice", F.col("ListPrice").cast(DoubleType()))
    .withColumn("Weight", F.col("Weight").cast(DoubleType()))
)

In [70]:
store_sales_order_header = (
    store_sales_order_header
    .withColumn("SalesOrderID", F.col("SalesOrderID").cast(IntegerType()))
    .withColumn("OrderDate", F.to_date("OrderDate"))
    .withColumn("ShipDate", F.to_date("ShipDate"))
    .withColumn("OnlineOrderFlag", F.col("OnlineOrderFlag").cast(BooleanType()))
    .withColumn("CustomerID", F.col("CustomerID").cast(IntegerType()))
    .withColumn("SalesPersonID", F.col("SalesPersonID").cast(IntegerType()))
    .withColumn("Freight", F.col("Freight").cast(DoubleType()))
)

In [72]:
store_sales_order_detail = (
    raw_sales_order_detail
    .withColumn("SalesOrderID", F.col("SalesOrderID").cast(IntegerType()))
    .withColumn("SalesOrderDetailID", F.col("SalesOrderDetailID").cast(IntegerType()))
    .withColumn("OrderQty", F.col("OrderQty").cast(IntegerType()))
    .withColumn("ProductID", F.col("ProductID").cast(IntegerType()))
    .withColumn("UnitPrice", F.col("UnitPrice").cast(DoubleType()))
    .withColumn("UnitPriceDiscount", F.col("UnitPriceDiscount").cast(DoubleType()))
)

In [74]:
#print schema to see schema with datatypes applied

store_product.printSchema()

store_sales_order_header.printSchema()

store_sales_order_detail.printSchema()

root
 |-- ProductID: integer (nullable = true)
 |-- ProductDesc: string (nullable = true)
 |-- ProductNumber: string (nullable = true)
 |-- MakeFlag: boolean (nullable = true)
 |-- Color: string (nullable = false)
 |-- SafetyStockLevel: integer (nullable = true)
 |-- ReorderPoint: integer (nullable = true)
 |-- StandardCost: double (nullable = true)
 |-- ListPrice: double (nullable = true)
 |-- Size: string (nullable = true)
 |-- SizeUnitMeasureCode: string (nullable = true)
 |-- Weight: double (nullable = true)
 |-- WeightUnitMeasureCode: string (nullable = true)
 |-- ProductCategoryName: string (nullable = true)
 |-- ProductSubCategoryName: string (nullable = true)

root
 |-- SalesOrderID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- ShipDate: date (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- Freight: 

## Data Quality Validation

The `store_` datasets have now been standardised and assigned the appropriate data types.

This section validates the quality and integrity of the standardised data before any business transformations are applied to the publish layer.

The following checks are performed:

- Primary key validation
- Mandatory field validation
- Referential integrity
- Date validation
- Numeric range validation
- Business rule validation

### Primary Key Validation

Each table is validated to ensure that the expected primary key uniquely identifies every record and does not contain NULL values.

In [77]:
def validate_primary_key(df, table_name, key_columns):
    """
    Validates that the primary key uniquely identifies each record
    and does not contain NULL values.
    """
    print(f"\n{'-' * 60}")
    print(f"Primary Key Validation: {table_name}")
    print(f"{'-' * 60}")

    # Group by the primary key and count the number of occurrences.
    # Any key appearing more than once indicates duplicate records.
    duplicates = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_count = duplicates.count()

    if duplicate_count == 0:
        print("There are NO duplicate primary keys found.")
    else:
        print(f"!! {duplicate_count} duplicate primary key(s) found.")
        duplicates.show(truncate=False)

    # Build a condition to check whether any of the primary key columns
    # contain NULL values. Primary keys should always be populated.
    null_condition = None

    for col in key_columns:
        condition = F.col(col).isNull()
        null_condition = (
            condition if null_condition is None
            else (null_condition | condition)
        )

    null_rows = df.filter(null_condition)

    null_count = null_rows.count()

    if null_count == 0:
        print("There are NO NULL primary keys found.")
    else:
        print(f"!! {null_count} NULL primary key(s) found.")
        null_rows.show(truncate=False)

In [79]:
validate_primary_key(store_product, "Product", ["ProductID"])

validate_primary_key(store_sales_order_header, "SalesOrderHeader", ["SalesOrderID"])

validate_primary_key(store_sales_order_detail, "SalesOrderDetail", ["SalesOrderDetailID"])


------------------------------------------------------------
Primary Key Validation: Product
------------------------------------------------------------
There are NO duplicate primary keys found.
There are NO NULL primary keys found.

------------------------------------------------------------
Primary Key Validation: SalesOrderHeader
------------------------------------------------------------
There are NO duplicate primary keys found.
There are NO NULL primary keys found.

------------------------------------------------------------
Primary Key Validation: SalesOrderDetail
------------------------------------------------------------
There are NO duplicate primary keys found.
There are NO NULL primary keys found.


### Referential Integrity

Relationships between the fact and dimension tables are validated to ensure that every foreign key references an existing parent record.

The following relationships are validated:

- SalesOrderDetail → SalesOrderHeader
- SalesOrderDetail → Product

In [81]:
#script for referential integrity

def validate_foreign_key(
    child_df,
    parent_df,
    child_key,
    parent_key,
    relationship_name
):
    """
    Validates that every foreign key value in the child table
    exists in the corresponding parent table.
    """

    print(f"\n{'-' * 60}")
    print(f"Referential Integrity: {relationship_name}")
    print(f"{'-' * 60}")

    # Perform a left anti join to identify child records
    # that do not have a matching parent record.
    orphan_rows = (
        child_df.alias("child")
        .join(
            parent_df.select(parent_key).distinct().alias("parent"),
            F.col(f"child.{child_key}") == F.col(f"parent.{parent_key}"),
            "left_anti"
        )
    )

    orphan_count = orphan_rows.count()

    if orphan_count == 0:
        print("There are NO referential integrity issues found.")
    else:
        print(f"!! {orphan_count} orphan record(s) found.")
        orphan_rows.show(truncate=False)

In [83]:
validate_foreign_key(
    store_sales_order_detail,
    store_sales_order_header,
    "SalesOrderID",
    "SalesOrderID",
    "SalesOrderDetail → SalesOrderHeader"
)

validate_foreign_key(
    store_sales_order_detail,
    store_product,
    "ProductID",
    "ProductID",
    "SalesOrderDetail → Product"
)


------------------------------------------------------------
Referential Integrity: SalesOrderDetail → SalesOrderHeader
------------------------------------------------------------
There are NO referential integrity issues found.

------------------------------------------------------------
Referential Integrity: SalesOrderDetail → Product
------------------------------------------------------------
There are NO referential integrity issues found.


**Observation**

No referential integrity issues were identified. Every SalesOrderID and ProductID referenced within the SalesOrderDetail table exists in the corresponding parent table.

## Primary and Foreign Keys Identification

Based on the data model and relationships between the tables, the following keys were identified.

| Table | Primary Key | Foreign Key(s) |
|------|-------------|----------------|
| `store_product` | `ProductID` | None |
| `store_sales_order_header` | `SalesOrderID` | None |
| `store_sales_order_detail` | `SalesOrderDetailID` | `SalesOrderID` → `store_sales_order_header.SalesOrderID`<br>`ProductID` → `store_product.ProductID` |

These relationships were validated during the data quality checks by confirming primary key uniqueness and verifying referential integrity between the parent and child tables.

## Date Validation

The standardised date columns are validated to ensure that shipment dates do not occur before the corresponding order dates.

In [86]:
store_sales_order_header.filter(
    F.col("ShipDate") < F.col("OrderDate")
).show(truncate=False)

+------------+---------+--------+---------------+-------------+----------+-------------+-------+
|SalesOrderID|OrderDate|ShipDate|OnlineOrderFlag|AccountNumber|CustomerID|SalesPersonID|Freight|
+------------+---------+--------+---------------+-------------+----------+-------------+-------+
+------------+---------+--------+---------------+-------------+----------+-------------+-------+



## Numeric Range Validation

Numeric measures are validated to ensure they fall within expected business ranges.

The following checks are performed:

- OrderQty > 0
- UnitPrice ≥ 0
- UnitPriceDiscount ≥ 0
- Freight ≥ 0

In [88]:
def validate_numeric_range(
    df,
    table_name,
    column_name,
    minimum_value=0,
    inclusive=True
):
    """
    Validates that numeric values satisfy the expected minimum value.

    Example:
        inclusive=True  -> value >= minimum_value
        inclusive=False -> value > minimum_value
    """

    print(f"\n{'-' * 60}")
    print(f"Numeric Range Validation: {table_name}.{column_name}")
    print(f"{'-' * 60}")

    # Apply the appropriate business rule.
    # For example:
    # OrderQty should be > 0
    # UnitPrice should be >= 0
    if inclusive:
        invalid_rows = df.filter(F.col(column_name) < minimum_value)
    else:
        invalid_rows = df.filter(F.col(column_name) <= minimum_value)

    invalid_count = invalid_rows.count()

    if invalid_count == 0:
        print("Validation passed.")
    else:
        print(f"!! {invalid_count} invalid record(s) found.")

        # Display the records that failed validation
        # to support investigation.
        invalid_rows.show(truncate=False)

In [90]:
# run functions

validate_numeric_range(
    store_sales_order_detail,
    "SalesOrderDetail",
    "OrderQty",
    minimum_value=0,
    inclusive=False      # Must be > 0
)

validate_numeric_range(
    store_sales_order_detail,
    "SalesOrderDetail",
    "UnitPrice"
)

validate_numeric_range(
    store_sales_order_detail,
    "SalesOrderDetail",
    "UnitPriceDiscount"
)

validate_numeric_range(
    store_sales_order_header,
    "SalesOrderHeader",
    "Freight"
)


------------------------------------------------------------
Numeric Range Validation: SalesOrderDetail.OrderQty
------------------------------------------------------------
!! 2 invalid record(s) found.
+------------+------------------+--------+---------+---------+-----------------+
|SalesOrderID|SalesOrderDetailID|OrderQty|ProductID|UnitPrice|UnitPriceDiscount|
+------------+------------------+--------+---------+---------+-----------------+
|43670       |112               |-1      |710      |5.7      |0.0              |
|43694       |339               |-1      |707      |20.1865  |0.0              |
+------------+------------------+--------+---------+---------+-----------------+


------------------------------------------------------------
Numeric Range Validation: SalesOrderDetail.UnitPrice
------------------------------------------------------------
Validation passed.

------------------------------------------------------------
Numeric Range Validation: SalesOrderDetail.UnitPric

#### Negative Order Quantity: Findings

A numeric range validation identified two records with an `OrderQty` of `-1`.

Negative quantities may represent legitimate business events, such as product returns or credit transactions. However, the provided data does not contain sufficient information to determine whether these values are intentional or the result of data entry errors.

As no business rules were provided to distinguish valid returns from invalid quantities, these records have been retained in the dataset. This avoids making unsupported assumptions while preserving the integrity of the source data.

## Record Count Reconciliation

A final record count reconciliation was performed to verify that the standardisation process produced the expected number of records and to ensure that no unintended data loss occurred.

The Product dataset contains fewer records in the `store_` layer than in the raw source. This reduction is expected and results from consolidating duplicate `ProductID` records identified during the data review. In each duplicate pair, the record containing populated product category information was retained.

The SalesOrderHeader and SalesOrderDetail datasets retain the same number of records as their corresponding raw datasets, confirming that no records were unintentionally removed during standardisation.

In [93]:
print("Record Count Reconciliation")
print("-" * 60)

row_counts = [
    ("Product", raw_product.count(), store_product.count()),
    ("SalesOrderHeader", raw_sales_order_header.count(), store_sales_order_header.count()),
    ("SalesOrderDetail", raw_sales_order_detail.count(), store_sales_order_detail.count())
]

print(f"{'Table':<20}{'Raw':>10}{'Store':>10}{'Difference':>15}")
print("-" * 60)

for table, raw_count, store_count in row_counts:
    difference = store_count - raw_count
    print(f"{table:<20}{raw_count:>10}{store_count:>10}{difference:>15}")

Record Count Reconciliation
------------------------------------------------------------
Table                      Raw     Store     Difference
------------------------------------------------------------
Product                    303       295             -8
SalesOrderHeader         31465     31465              0
SalesOrderDetail        121317    121317              0


## Summary

The raw datasets were reviewed, cleaned, and stored as the following tables:

- `store_product`
- `store_sales_order_header`
- `store_sales_order_detail`

The following activities were completed during the data review and storage stage:

- Reviewed the source data and assigned appropriate data types to each column.
- Identified the primary and foreign keys for each table based on the data model.
- Assessed missing values, duplicate records, and date and numeric formatting issues.
- Resolved duplicate `ProductID` records by retaining the record with the more complete category information.
- Converted malformed date values (for example, `2021-06`) to `NULL` rather than making assumptions about the intended date.
- Standardised the data by casting columns to their appropriate data types.
- Performed data quality validation, including primary key checks, referential integrity, date logic, numeric range validation, and business rule validation.
- Retained the two records with negative `OrderQty` values and flagged them as potential data quality issues, as the source data did not provide enough information to determine whether they represented returns or data entry errors.

The cleaned datasets produced in this stage provide the foundation for the business transformations and analysis completed in the subsequent sections.

**The next section is the publish layer where the business transformations specified in the assessment to produce datasets suitable for downstream reporting and analysis are applied.**


# 3. Product Master Transformations

The Product dataset is enriched according to the business rules provided in the assessment.

The following transformations are applied:

- Replace missing `Color` values with `"N/A"`.
- Populate missing `ProductCategoryName` values using the corresponding `ProductSubCategoryName`.

In [98]:
publish_product = (
    store_product

    # Replace missing colours with the required default value
    .fillna({"Color": "N/A"})

    # Populate missing ProductCategoryName using the
    # business rules provided in the assessment
    .withColumn(
        "ProductCategoryName",
        F.when(
            F.col("ProductCategoryName").isNotNull(),
            F.col("ProductCategoryName")
        )
        .when(
            F.col("ProductSubCategoryName").isin(
                "Gloves",
                "Shorts",
                "Socks",
                "Tights",
                "Vests"
            ),
            F.lit("Clothing")
        )
        .when(
            F.col("ProductSubCategoryName").isin(
                "Locks",
                "Lights",
                "Headsets",
                "Helmets",
                "Pedals",
                "Pumps"
            ),
            F.lit("Accessories")
        )
        .when(
            (F.col("ProductSubCategoryName").contains("Frames")) |
            (F.col("ProductSubCategoryName").isin("Wheels", "Saddles")),
            F.lit("Components")
        )
        .otherwise(F.col("ProductCategoryName"))
    )
)

In [102]:
publish_product.show(5, truncate=False)

+---------+-------------------------+-------------+--------+-----+----------------+------------+------------+---------+----+-------------------+------+---------------------+-------------------+----------------------+
|ProductID|ProductDesc              |ProductNumber|MakeFlag|Color|SafetyStockLevel|ReorderPoint|StandardCost|ListPrice|Size|SizeUnitMeasureCode|Weight|WeightUnitMeasureCode|ProductCategoryName|ProductSubCategoryName|
+---------+-------------------------+-------------+--------+-----+----------------+------------+------------+---------+----+-------------------+------+---------------------+-------------------+----------------------+
|680      |HL Road Frame - Black, 58|FR-R92B-58   |true    |Black|500             |375         |1059.31     |1431.5   |58  |CM                 |2.24  |LB                   |Components         |Road Frames           |
|706      |HL Road Frame - Red, 58  |FR-R92R-58   |true    |Red  |500             |375         |1059.31     |1431.5   |58  |CM      

#### *Transformation Validation*

In [ ]:
#check logic transformation was implemented for Color column

publish_product.filter(
    F.col("Color").isNull()
).show()

publish_product.filter(
    F.col("Color").isNull()
).count()

In [109]:
# Showing N/A was applied

publish_product.filter(
    F.col("Color") == "N/A"
).show(5, truncate=False)


+---------+-----------+-------------+--------+-----+----------------+------------+------------+---------+----+-------------------+------+---------------------+-------------------+----------------------+
|ProductID|ProductDesc|ProductNumber|MakeFlag|Color|SafetyStockLevel|ReorderPoint|StandardCost|ListPrice|Size|SizeUnitMeasureCode|Weight|WeightUnitMeasureCode|ProductCategoryName|ProductSubCategoryName|
+---------+-----------+-------------+--------+-----+----------------+------------+------------+---------+----+-------------------+------+---------------------+-------------------+----------------------+
|802      |LL Fork    |FK-1639      |true    |N/A  |500             |375         |65.8097     |148.22   |NULL|NULL               |NULL  |NULL                 |NULL               |Forks                 |
|803      |ML Fork    |FK-5136      |true    |N/A  |500             |375         |77.9176     |175.49   |NULL|NULL               |NULL  |NULL                 |NULL               |Forks    

In [111]:
# check product category transformation

publish_product.filter(
    F.col("ProductSubCategoryName").isin(
        "Gloves",
        "Shorts",
        "Socks",
        "Tights",
        "Vests",
        "Locks",
        "Lights",
        "Headsets",
        "Helmets",
        "Pedals",
        "Pumps",
        "Wheels",
        "Saddles"
    )
    |
    F.col("ProductSubCategoryName").contains("Frames")
).select(
    "ProductID",
    "ProductSubCategoryName",
    "ProductCategoryName"
).show(truncate=False)

+---------+----------------------+-------------------+
|ProductID|ProductSubCategoryName|ProductCategoryName|
+---------+----------------------+-------------------+
|680      |Road Frames           |Components         |
|706      |Road Frames           |Components         |
|707      |Helmets               |Accessories        |
|708      |Helmets               |Accessories        |
|709      |Socks                 |Clothing           |
|710      |Socks                 |Clothing           |
|711      |Helmets               |Accessories        |
|717      |Road Frames           |Components         |
|718      |Road Frames           |Components         |
|719      |Road Frames           |Components         |
|720      |Road Frames           |Components         |
|721      |Road Frames           |Components         |
|722      |Road Frames           |Components         |
|723      |Road Frames           |Components         |
|724      |Road Frames           |Components         |
|725      

In [113]:
# Check how many records met the criteria for the category transformation

eligible_store = (
    store_product.filter(
        (F.col("ProductCategoryName").isNull()) &
        (
            F.col("ProductSubCategoryName").isin(
                "Gloves", "Shorts", "Socks", "Tights", "Vests",
                "Locks", "Lights", "Headsets", "Helmets", "Pedals", "Pumps",
                "Wheels", "Saddles"
            ) |
            F.col("ProductSubCategoryName").contains("Frames")
        )
    ).count()
)

remaining_publish = (
    publish_product.filter(
        (F.col("ProductCategoryName").isNull()) &
        (
            F.col("ProductSubCategoryName").isin(
                "Gloves", "Shorts", "Socks", "Tights", "Vests",
                "Locks", "Lights", "Headsets", "Helmets", "Pedals", "Pumps",
                "Wheels", "Saddles"
            ) |
            F.col("ProductSubCategoryName").contains("Frames")
        )
    ).count()
)


print(f"Eligible records before transformation : {eligible_store}")
print(f"Eligible records NOT updated           : {remaining_publish}")

Eligible records before transformation : 144
Eligible records NOT updated           : 0


**There were 144 records with a missing ProductCategoryName that met the business rules. After applying the transformation, all 144 records were successfully populated.**

### Transformation Validation

The transformed Product dataset was reviewed to verify that:

- Missing `Color` values were replaced with `"N/A"`.
- Missing `ProductCategoryName` values were populated according to the specified business rules.
- Existing category values were preserved and not overwritten.

# 4. Sales Order Transformations

The SalesOrderDetail and SalesOrderHeader datasets are joined on `SalesOrderID` to create the `publish_orders` dataset.

The following business transformations are applied:

- Calculate **LeadTimeInBusinessDays** as the number of business days between the order date and ship date, excluding Saturdays and Sundays.
- Calculate **TotalLineExtendedPrice** using the formula:

> `OrderQty × (UnitPrice − UnitPriceDiscount)`

After joining the tables, the final publish_orders dataset contains every column from SalesOrderDetail together with the relevant columns from SalesOrderHeader. The duplicate join key (SalesOrderID) is included only once, and the Freight column is renamed to TotalOrderFreight to match the assessment requirements.

### Join the tables

In [117]:
# Join the standardised Sales Order Header and Detail tables.
# An inner join is appropriate because every order detail should have a corresponding order header.


orders_df = (
    store_sales_order_detail.alias("detail")
    .join(
        store_sales_order_header.alias("header"),
        on="SalesOrderID",
        how="inner"
    )
)

In [ ]:
orders_df.toPandas()

### Calculate LeadTimeInBusinessDays

In [119]:
# Calculate the number of business days between OrderDate and ShipDate.
# Spark generates every date in the range, removes weekends, then counts the remaining weekdays.

orders_df = (
    orders_df.withColumn(
        "LeadTimeInBusinessDays",
        F.size(
            F.filter(
                F.sequence(
                    F.col("OrderDate"),
                    F.col("ShipDate")
                ),
                lambda d: F.dayofweek(d).between(2, 6)
            )
        ) - 1
    )
)

### Calculate TotalLineExtendedPrice

In [121]:
# Calculate the extended line price for each order line.

orders_df = (
    orders_df.withColumn(
        "TotalLineExtendedPrice",
        F.col("OrderQty") *
        (
            F.col("UnitPrice") -
            F.col("UnitPriceDiscount")
        )
    )
)

### Rename Freight

In [123]:
# Rename Freight to match the assessment requirements.

orders_df = (
    orders_df.withColumnRenamed(
        "Freight",
        "TotalOrderFreight"
    )
)

In [125]:
orders_df.printSchema()

root
 |-- SalesOrderID: integer (nullable = true)
 |-- SalesOrderDetailID: integer (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitPriceDiscount: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- ShipDate: date (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TotalOrderFreight: double (nullable = true)
 |-- LeadTimeInBusinessDays: integer (nullable = true)
 |-- TotalLineExtendedPrice: double (nullable = true)



### Create Publish Orders Dataset

The final dataset contains:

- All columns from **SalesOrderDetail**
- All columns from **SalesOrderHeader** (excluding the duplicate `SalesOrderID`)
- The calculated business metrics

In [128]:
publish_orders = orders_df.select(

    # Detail columns
    *store_sales_order_detail.columns,

    # Header columns
    "OrderDate",
    "ShipDate",
    "OnlineOrderFlag",
    "AccountNumber",
    "CustomerID",
    "SalesPersonID",
    "TotalOrderFreight",

    # Calculated columns
    "LeadTimeInBusinessDays",
    "TotalLineExtendedPrice"
)


#### *Validating the output....*

A sample of the published dataset is displayed to confirm that the join and business transformations have been applied successfully.

In [132]:
publish_orders.show(5, truncate=False)

+------------+------------------+--------+---------+---------+-----------------+----------+----------+---------------+--------------+----------+-------------+-----------------+----------------------+----------------------+
|SalesOrderID|SalesOrderDetailID|OrderQty|ProductID|UnitPrice|UnitPriceDiscount|OrderDate |ShipDate  |OnlineOrderFlag|AccountNumber |CustomerID|SalesPersonID|TotalOrderFreight|LeadTimeInBusinessDays|TotalLineExtendedPrice|
+------------+------------------+--------+---------+---------+-----------------+----------+----------+---------------+--------------+----------+-------------+-----------------+----------------------+----------------------+
|43659       |1                 |1       |776      |2024.994 |0.0              |2021-05-31|2021-06-07|false          |10-4020-000676|29825     |279          |616.0984         |5                     |2024.994              |
|43659       |2                 |3       |777      |2024.994 |0.0              |2021-05-31|2021-06-07|false 

In [134]:
publish_orders.printSchema()

root
 |-- SalesOrderID: integer (nullable = true)
 |-- SalesOrderDetailID: integer (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitPriceDiscount: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- ShipDate: date (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TotalOrderFreight: double (nullable = true)
 |-- LeadTimeInBusinessDays: integer (nullable = true)
 |-- TotalLineExtendedPrice: double (nullable = true)



# Analysis Questions

The following analysis is performed on the transformed datasets:

- `publish_orders`
- `publish_product`

The goal is to derive insights from the final curated datasets.


### Analysis 1 

#### **Question:** Which color generated the highest revenue each year?

The order dataset contains the revenue for each order line, while the product dataset contains the product colour. These datasets are joined on `ProductID` so that revenue can be analysed by product colour.

Records with a `NULL` `OrderDate` are excluded from this analysis because a valid year cannot be derived from them. These records originated from malformed dates identified during the data quality review and cannot be assigned to a specific year without making assumptions.

In [136]:
# Join the order and product datasets so each order line has its corresponding product colour.
# Exclude records with a NULL OrderDate as they cannot be assigned to a year.


color_revenue_df = (
    publish_orders
    .filter(F.col("OrderDate").isNotNull())
    .join(
        publish_product.select("ProductID", "Color"),
        on="ProductID",
        how="left"
    )
    .withColumn("Year", F.year("OrderDate"))
)

# Calculate total revenue for each colour within each year.
color_revenue_agg = (
    color_revenue_df
    .groupBy("Year", "Color")
    .agg(
        F.sum("TotalLineExtendedPrice").alias("TotalRevenue")
    )
)

# Rank colours by revenue within each year.
window = (
    Window
    .partitionBy("Year")
    .orderBy(F.desc("TotalRevenue"))
)

ranked_colors = (
    color_revenue_agg
    .withColumn("rank", F.row_number().over(window))
)

# Keep only the highest revenue-generating colour for each year.
top_color_each_year = (
    ranked_colors
    .filter(F.col("rank") == 1)
    .drop("rank")
)

print("Q1 -- Highest revenue color per year:")
top_color_each_year.show()

Q1 -- Highest revenue color per year:
+----+------+--------------------+
|Year| Color|        TotalRevenue|
+----+------+--------------------+
|2021|   Red|   6005300.935699883|
|2022| Black|1.4005242975200394E7|
|2023| Black|1.5047694369201014E7|
|2024|Yellow|   6368158.478900213|
+----+------+--------------------+



### Analysis 2

#### **Question: What is the average LeadTimeInBusinessDays by ProductCategoryName**

The published orders dataset contains the lead time for each order line, while the product dataset contains the product category. The two datasets are joined on `ProductID` so that the average lead time can be calculated for each product category.

The results are grouped by `ProductCategoryName` and sorted in descending order to identify the categories with the longest average lead times.

In [138]:
lead_time_df = (
    publish_orders
    .join(
        publish_product.select("ProductID", "ProductCategoryName"),
        on="ProductID",
        how="left"
    )
)

avg_lead_time = (
    lead_time_df
    .groupBy("ProductCategoryName")
    .agg(
        F.avg("LeadTimeInBusinessDays").alias("AvgLeadTimeInBusinessDays")
    )
    .orderBy(F.desc("AvgLeadTimeInBusinessDays"))
)

avg_lead_time.show()

+-------------------+-------------------------+
|ProductCategoryName|AvgLeadTimeInBusinessDays|
+-------------------+-------------------------+
|               NULL|       4.7202082171407325|
|           Clothing|        4.709380234505863|
|        Accessories|        4.702787804316105|
|              Bikes|        4.666345536287733|
|         Components|        4.664859191883855|
+-------------------+-------------------------+



Filter out where `ProductCategoryName` is NULL

In [140]:
avg_lead_time = (
    avg_lead_time
    .filter(F.col("ProductCategoryName").isNotNull())
)

print("Q2 -- Average LeadTimeInBusinessDays by ProductCategoryName:")
avg_lead_time.show()

Q2 -- Average LeadTimeInBusinessDays by ProductCategoryName:
+-------------------+-------------------------+
|ProductCategoryName|AvgLeadTimeInBusinessDays|
+-------------------+-------------------------+
|           Clothing|        4.709380234505863|
|        Accessories|        4.702787804316105|
|              Bikes|        4.666345536287733|
|         Components|        4.664859191883855|
+-------------------+-------------------------+

